# 05 - Paired headline generation: full run

Writes a **sarcastic** and a **serious** headline for every seed headline of the training split (22,800 seeds by default), using three open LLMs (Qwen, Gemma, Mistral; each seed goes to one model, both sides from that model). Then checks every pair, picks a judge on real labelled training headlines (SmolLM3-3B, or cross-voting by the two generators that did not write the pair; none if neither is accurate enough), and writes the **frozen dataset** for Pass 4 training with SHA-256 fingerprints.

**Two stages, set in step 2 (`STAGE`):**
1. **`"generate"`**: generation, saved in chunks of 500 seeds (`raw_chunks/<model>/chunk_XXXX.csv`). Kaggle stops a session after 12 hours, so the full run is split into sessions; finished chunks are never redone.
2. **`"finalize"`**: collects every chunk, runs the checks and the judge contest, writes the dataset. About 1-2 hours (four models judge).

**First, a trial (strongly recommended, about 40 minutes):** set `N_SEEDS = 200`, `STAGE = "generate"`, `RUN_MODELS = ["qwen", "gemma", "mistral"]`, Run All, then `STAGE = "finalize"`, Run All. Check the tables (step 6) and the printed time per chunk. Then set `N_SEEDS = None` for the real run. Chunks from the trial are not reused (a different seed plan), so start the real run in a fresh session.

**Real run, one model per session** (time per model is printed per chunk; if one model takes more than about 10 hours, it simply continues in the next session):
1. Set `STAGE = "generate"`, `RUN_MODELS = ["qwen"]`, then **Save Version > Save & Run All (Commit)**. It runs in the background and keeps its output.
2. When it finishes: **Add Input > Your Work >** this notebook's output. Step 5 copies the chunks found under `/kaggle/input` into the working folder, so nothing is redone. Set `RUN_MODELS = ["gemma"]`, commit again. Same for `["mistral"]`.
3. Last: `STAGE = "finalize"` with the latest output added as input, commit. Download the result.

**Settings in the right-hand panel:** GPU T4 x2, Internet on, input **`gainsplit-processed`**, secret **`HF_TOKEN`** (Gemma only). Free: no API key, no paid service.

## 1. Get the code

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_pairgen.py", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 2. Settings
Change only `STAGE`, `RUN_MODELS` and `N_SEEDS` between sessions. Everything else must stay the same for the whole run, or the seed plan changes.

In [ ]:
from pathlib import Path

STAGE = "generate"                         # "generate" or "finalize"
RUN_MODELS = ["qwen", "gemma", "mistral"]  # models to generate in this session
N_SEEDS = None                             # None = whole training split; 200 for the trial

SEED = 2026
CHUNK = 500
MODELS = {
    "qwen":    {"id": "Qwen/Qwen2.5-7B-Instruct",           "extra": {}},
    "gemma":   {"id": "google/gemma-2-9b-it",               "extra": {"attn_implementation": "eager"}},
    "mistral": {"id": "mistralai/Mistral-7B-Instruct-v0.3", "extra": {}},
}
GEN = {"max_new_tokens": 120, "temperature": 0.8, "top_p": 0.95, "batch_size": 32}
SMOL = {"id": "HuggingFaceTB/SmolLM3-3B", "extra": {}}   # judge candidate from a family not used for generation
JUDGE_BATCH = 64
N_REAL_CALIB = 400       # real train headlines (200 per class) on which the judge candidates are scored
JUDGE_MIN = 0.75         # a candidate must reach this balanced accuracy to be used as the filter

OUT = Path("/kaggle/working/pairgen")
RAW = OUT / "raw_chunks"
RAW.mkdir(parents=True, exist_ok=True)

## 3. Data and the seed plan
Reads **only the training split**. The plan (which seed goes to which model) depends only on the train file, `SEED` and `MODELS`, so every session rebuilds the same plan. A plan file from an earlier session is compared with it and the run stops if they differ.

In [ ]:
import json, hashlib
import numpy as np
import pandas as pd
from gainsplit.data import find_data_dir
from gainsplit import pairgen as pg

DATA_DIR = find_data_dir("/kaggle/input")
train = pd.read_csv(DATA_DIR / "headlines_train.csv", keep_default_na=False)
plan = pg.plan_seeds(train, list(MODELS), SEED, N_SEEDS)
plan_hash = hashlib.sha256(pd.util.hash_pandas_object(plan, index=False).values.tobytes()).hexdigest()[:16]
print("data:", DATA_DIR, "| seeds:", len(plan), "| plan fingerprint:", plan_hash)

old = list(Path("/kaggle/input").rglob("pairgen/plan_fingerprint.txt")) + [OUT / "plan_fingerprint.txt"]
for f in old:
    if f.exists():
        assert f.read_text().strip() == plan_hash, f"seed plan differs from {f}: same N_SEEDS, SEED and MODELS as before?"
(OUT / "plan_fingerprint.txt").write_text(plan_hash)
plan.to_csv(OUT / "seed_plan.csv", index=False)
print(pd.crosstab(plan["model"], plan["seed_label"], margins=True))

## 4. Secrets, GPUs, model loading

In [ ]:
import gc, shutil, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_TOKEN = get_secret("HF_TOKEN")
print("HF_TOKEN found:", HF_TOKEN is not None)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def load(model_id, extra):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, token=HF_TOKEN, torch_dtype=torch.float16,
                                                 device_map="auto", **extra)
    model.eval()
    return tok, model

def run_prompts(tok, model, prompts, batch_size, max_new_tokens, sample):
    outs = []
    for i in range(0, len(prompts), batch_size):
        texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
                 for p in prompts[i:i + batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        kw = dict(do_sample=True, temperature=GEN["temperature"], top_p=GEN["top_p"]) if sample else dict(do_sample=False)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

## 5. Generate (`STAGE = "generate"`)
First copies chunks saved by earlier sessions (added as input) into the working folder. Then, for each model in `RUN_MODELS`, generates every chunk that is not saved yet, saving after each chunk and printing the time and the estimated time left for that model. Sampling is seeded per chunk, so a chunk is reproducible on the same hardware.

In [ ]:
if STAGE == "generate":
    copied = 0
    for p in pg.find_chunks("/kaggle/input"):
        dest = RAW / p.parent.name / p.name
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(p, dest); copied += 1
    print("chunks copied from earlier sessions:", copied)

    log_path = OUT / "timing.csv"
    for name in RUN_MODELS:
        chunks = pg.model_chunks(plan, name, CHUNK)
        todo = [k for k in range(len(chunks)) if not (RAW / name / pg.chunk_name(k)).exists()]
        print(f"\n{name}: {len(chunks)} chunks, {len(todo)} to do")
        if not todo:
            continue
        t0 = time.time()
        tok, model = load(MODELS[name]["id"], MODELS[name]["extra"])
        print(f"loaded in {time.time() - t0:.0f} s")
        (RAW / name).mkdir(parents=True, exist_ok=True)
        for n_done, k in enumerate(todo, 1):
            part = chunks[k].copy()
            torch.manual_seed(SEED * 10_000 + k)
            t1 = time.time()
            part["raw"] = run_prompts(tok, model, [pg.build_prompt(t) for t in part["seed_text"]],
                                      GEN["batch_size"], GEN["max_new_tokens"], sample=True)
            secs = time.time() - t1
            part.to_csv(RAW / name / pg.chunk_name(k), index=False)
            pd.DataFrame([{"model": name, "chunk": k, "seeds": len(part), "seconds": round(secs, 1)}]).to_csv(
                log_path, mode="a", header=not log_path.exists(), index=False)
            left = secs * (len(todo) - n_done) / 3600
            print(f"  chunk {k}: {len(part)} seeds in {secs:.0f} s | about {left:.1f} h left for {name}")
        del model, tok; gc.collect(); torch.cuda.empty_cache()
    print("\nsaved chunks:", {m: len(list((RAW / m).glob('chunk_*.csv'))) for m in MODELS if (RAW / m).exists()})
else:
    print("STAGE is", STAGE, "- generation skipped")

## 6. Finalize (`STAGE = "finalize"`): checks
Collects all chunks (from this session and from inputs), stops if any seed of the plan is missing, then runs the automatic checks (refusal, valid output, empty, too long, identical sides, copy of the seed) and the outlet-cue table: share of headlines containing one of the 100 words that most strongly mark each class in the real data. The sarcastic side should not be much richer in Onion cue words than the real sarcastic headlines.

In [ ]:
if STAGE == "finalize":
    paths = pg.find_chunks(OUT) + pg.find_chunks("/kaggle/input")
    raw = pg.collect_raw(paths)
    raw = raw[raw["seed_id"].isin(plan["seed_id"])]
    missing = len(plan) - len(raw)
    print("seeds generated:", len(raw), "| missing:", missing)
    assert missing == 0, "some chunks are missing: run STAGE='generate' for the models with chunks left"

    checks = pd.DataFrame([pg.check_pair(s, r) for s, r in zip(raw["seed_text"], raw["raw"])])
    pairs = pd.concat([raw.reset_index(drop=True), checks], axis=1)
    flags = ["refusal", "parse_ok", "empty", "too_long", "same_pair", "copies_seed", "ok"]
    display(pairs.groupby("model")[flags].mean().round(3))

    cues = json.loads((DATA_DIR / "cue_words.json").read_text())
    sc, nc = [w for w, _ in cues["sarcastic"]], [w for w, _ in cues["non_sarcastic"]]
    good = pairs[pairs["ok"]]
    rows = [{"source": f"{m}: {side}", "sarc_cues": pg.cue_rate(g[side], sc), "serious_cues": pg.cue_rate(g[side], nc)}
            for m, g in good.groupby("model") for side in ("sarcastic", "serious")]
    rows += [{"source": f"real train: label {lab}", "sarc_cues": pg.cue_rate(train.loc[train["label"] == lab, "text"], sc),
              "serious_cues": pg.cue_rate(train.loc[train["label"] == lab, "text"], nc)} for lab in (1, 0)]
    display(pd.DataFrame(rows).round(3))

## 7. Finalize: judge contest and the frozen dataset
**Which judge?** Decided on real data, not by us. Two candidates label 400 real training headlines whose labels are known (200 per class):
- **`smollm3`**: Hugging Face's SmolLM3-3B, a family not used for generation.
- **`cross`**: the two generator models that did *not* write a pair judge it; a verdict counts only when both give the same answer. On the real headlines every pair of generators is scored and the three scores are averaged.

The score is balanced accuracy (the mean of the accuracy on sarcastic and on serious headlines; an unclear answer counts as wrong). The best candidate becomes the judge if it reaches `JUDGE_MIN` (0.75). If none does, **no judge filters**: every pair that passed the checks is kept, and the report says so.

Every judge sees each headline alone (no pair, no hint), with greedy decoding. A pair is **kept** when it passed the checks and, if there is a judge, the judge agrees with both sides. Writes:
- `pairs_all.csv`: every seed, raw output, checks, `kept`.
- `pairs_train.csv`: training rows (`id`, `seed_id`, `model`, `text`, `label`) from kept pairs only. **This is the file Pass 4 trains on.**
- `votes.csv`: every judge's answer for every headline; `manifest_pairs.json`: contest scores, chosen judge, counts, settings, SHA-256 of each file.

In [ ]:
if STAGE == "finalize":
    items = pg.long_items(good)
    calib = pg.pick_seeds(train, N_REAL_CALIB, SEED + 1)
    real = pd.DataFrame({"item_id": calib["seed_id"].astype(str) + "_real", "seed_id": calib["seed_id"], "model": "real",
                         "intended": calib["seed_label"], "text": calib["seed_text"]})
    to_judge = pd.concat([real, items], ignore_index=True)
    print("real headlines:", len(real), "| generated headlines:", len(items))

    judges = {"smollm3": SMOL, **MODELS}
    votes = []
    for jname, spec in judges.items():
        sub = to_judge if jname == "smollm3" else to_judge[to_judge["model"] != jname]   # no model judges its own text
        t0 = time.time()
        tok, model = load(spec["id"], spec["extra"])
        answers = run_prompts(tok, model, [pg.judge_prompt(t) for t in sub["text"]], JUDGE_BATCH, 16, sample=False)
        del model, tok; gc.collect(); torch.cuda.empty_cache()
        votes.append(pd.DataFrame({"item_id": sub["item_id"].values, "judge": jname,
                                   "raw": [a.strip()[:40] for a in answers], "pred": [pg.parse_judge(a) for a in answers]}))
        print(f"{jname}: {len(sub)} headlines in {(time.time() - t0) / 60:.1f} min, unclear {votes[-1]['pred'].isna().mean():.3f}")
    votes = pd.concat(votes, ignore_index=True)
    votes.to_csv(OUT / "votes.csv", index=False)
    wide = votes.pivot(index="item_id", columns="judge", values="pred")

    # contest on the real headlines
    r = wide.loc[real["item_id"]]
    scores_detail = {"smollm3": pg.balanced_accuracy(real["intended"], r["smollm3"])}
    for m in MODELS:
        others = [g for g in MODELS if g != m]
        scores_detail[f"cross without {m}"] = pg.balanced_accuracy(real["intended"], [pg.cross_verdict(v) for v in r[others].values])
    scores = {"smollm3": scores_detail["smollm3"],
              "cross": float(np.mean([scores_detail[f"cross without {m}"] for m in MODELS]))}
    for name, v in scores_detail.items():
        print(f"  {name:22s} balanced accuracy on real headlines: {v:.3f}")
    JUDGE = pg.pick_judge(scores, JUDGE_MIN)
    print("chosen judge:", JUDGE or f"none (best {max(scores.values()):.3f} < {JUDGE_MIN}): pairs are not filtered by a judge")

    # verdicts on the generated headlines from the chosen judge
    def verdict(row):
        if JUDGE == "smollm3":
            return pg._pred(wide.at[row.item_id, "smollm3"])
        others = [g for g in MODELS if g != row.model]
        return pg.cross_verdict(wide.loc[row.item_id, others].values)
    if JUDGE:
        items["verdict"] = [verdict(rw) for rw in items.itertuples()]
        items["agrees"] = items["verdict"] == items["intended"]
        display(items.assign(side=items["intended"].map({1: "sarcastic", 0: "serious"}))
                .groupby(["model", "side"]).agg(n=("agrees", "size"), judge_agrees=("agrees", "mean")).round(3))
    else:
        items["verdict"], items["agrees"] = None, True
    both = items.groupby("seed_id")["agrees"].agg(lambda s: bool(len(s) == 2 and s.all()))
    pairs["judge_both"] = pairs["seed_id"].map(both).fillna(False).astype(bool)
    pairs["kept"] = pairs["ok"] & pairs["judge_both"]
    display(pairs.groupby("model").agg(seeds=("kept", "size"), ok=("ok", "mean"), kept=("kept", "mean")).round(3))
    display(pairs.pivot_table(index="model", columns="seed_label", values="kept", aggfunc="mean").round(3))

    pairs.to_csv(OUT / "pairs_all.csv", index=False)
    rows_train = pg.training_rows(pairs)
    rows_train.to_csv(OUT / "pairs_train.csv", index=False)

    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    files = ["pairs_all.csv", "pairs_train.csv", "votes.csv", "seed_plan.csv"]
    manifest = {
        "commit": COMMIT, "plan_fingerprint": plan_hash, "seed": SEED, "n_seeds": len(plan), "models": MODELS,
        "prompt_version": pg.PROMPT_VERSION, "gen": GEN, "max_words": pg.MAX_WORDS,
        "judge_contest": {"candidates": {"smollm3": SMOL["id"], "cross": "the two generators that did not write the pair"},
                          "real_headlines": len(real), "scores": {k: round(v, 3) for k, v in scores_detail.items()},
                          "min_score": JUDGE_MIN, "chosen": JUDGE},
        "counts": {"pairs_ok": int(pairs["ok"].sum()), "pairs_kept": int(pairs["kept"].sum()),
                   "training_rows": len(rows_train),
                   "kept_by_model": pairs.groupby("model")["kept"].sum().astype(int).to_dict()},
        "sha256": {f: sha(OUT / f) for f in files},
    }
    (OUT / "manifest_pairs.json").write_text(json.dumps(manifest, indent=2, default=str))
    print(json.dumps({k: v for k, v in manifest.items() if k != "sha256"}, indent=2, default=str))

## 8. Download
Zips the output folder (the raw chunks too). Download `pairgen.zip` and keep it in `results/pairgen/` locally; `pairs_train.csv` and `manifest_pairs.json` later go into a new version of `gainsplit-processed` for Pass 4. Send the tables from steps 6-7 and the printed manifest.

In [ ]:
import shutil
print(shutil.make_archive("/kaggle/working/pairgen", "zip", OUT))